# <center>企业级知识中台·第四节课：Docker Compose 部署、Neo4j 只读观察</center>

&emsp;&emsp;前三节里，我们已经分别搭起了 RAG、GraphRAG 和 Agent 的关键能力：知识能够被检索，关系能够被组织，请求也能够被编排。但如果这些能力还停留在本地单独运行的脚本里，就还没有回答一个真实系统的问题——浏览器中的一次资料读取或 Agent 请求，究竟经过了哪些服务、哪些数据边界和哪些权限判断？

&emsp;&emsp;这一节把镜头从单个能力拉回到完整系统。我们会先建立 Docker Compose（用一份配置协同启动多个容器服务的工具）的运行地图，理解为什么默认只开放 Web；再进入 Neo4j Browser（Neo4j 提供的图查询与可视化界面）观察 GraphRAG 的关系；随后在 DBeaver 中执行受控的 `SELECT`，查看业务状态、用户身份、权限判断和真实可读数据。

&emsp;&emsp;这不是一份 Docker 命令清单，也不是一次数据库导出练习。每个操作都只为回答一个具体问题：系统是否真正启动？数据为什么分别放在两类存储？当前身份为什么能读、不能管，或不能打开私人会话？当你能沿这条链找到证据，才能在自己的环境中复现和定位问题。

> 📌 **目标受众与前置要求**：本节面向已完成前三节学习、能够阅读 Python、Docker Compose 和基础 SQL 的开发者。你需要 Docker Desktop、项目源码和自己的私有 `deploy/compose/.env`（保存本机私有运行配置的文件）；不需要在宿主机单独安装 PostgreSQL、Neo4j、Bun（JavaScript 运行时）或三条 RAG 的运行时。

> 📌 **学完本节你将带走 5 件产物**：① 一张 Compose 服务与端口边界记录；② 一次 Neo4j Browser 的局部子图观察；③ 一次 DBeaver 连接与会话只读确认；④ 一次从 username 到 `user_id` 的身份查询；⑤ 一张同时展示 `can_read`、`can_manage` 和真实可读数据的权限查询结果。

> **【本节边界】**：本节使用当前 Docker 数据库中已有的 A/B/admin 脱敏教学账号和 P/T/C 场景观察用户资源权限；三条 RAG 的真实摄取、Agent 端到端调用、更多身份隔离探针和 Windows 冷启动仍需要额外运行证据，不在本节中写成已通过。

> 📅 **时效性说明**：本节依据当前 `ff-companybrain/` 的 Compose 配置、源码与已记录的本地 Docker 基线编写。端口、数据量与外部模型的实际响应应以你自己的运行输出为准；不要把示例中的任何私有配置、账号或业务内容复制到截图和笔记中。

> **【术语阅读约定】**：服务名、字段名和命令保持源码原样；第一次出现时用括注说明它在本节中的职责，后文不重复展开。

## <center>第一章：系统运行边界与存储职责</center>

&emsp;&emsp;前三节课解决的是“知识能力怎样成立”：传统 RAG 负责从文档中检索证据，GraphRAG 负责组织实体与关系，Agent Gateway 负责把检索、会话和工具调用编排起来。第四节课不再新增一条孤立能力，而是把这些能力放回同一套正在运行的系统中，回答“浏览器中的一次操作究竟经过了什么”。

&emsp;&emsp;这一章是全课唯一的纯背景章，没有代码。我们先区分代码仓库、容器服务、数据存储和观察入口，再建立 PostgreSQL 与 Neo4j 的职责地图。地图建立之后，第二章开始部署；第三、四章分别进入图数据库与关系数据库；第五章再从真实 `user_id` 查询用户权限和可读数据。

&emsp;&emsp;最容易出现的误解有三个：代码在本地不等于系统已经运行；容器健康不等于 RAG 摄取和 Agent 调用已经通过；数据库能连接也不等于当前用户拥有业务权限。后面每一步都会给出相应证据，避免用一个绿色状态替代整条链路的结论。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171625009.png" alt="企业知识中台的 Docker 运行边界与数据流" width=78%></div>

### 1.1 系统运行证据链

&emsp;&emsp;前三节课留下的代码和知识不会失效，但运行方式发生了变化。过去我们可以在本地进程里单独观察一条链路；现在 Web、API、Agent Gateway、三条 RAG、PostgreSQL 与 Neo4j 由 Docker Compose 共同管理。我们面对的不再是“运行一个脚本”，而是“确认一组服务是否按边界协作”。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>前三节能力与本节系统问题</font></p>
<div class="center">

| 已有能力 | 单独运行时能回答什么 | 放进完整系统后还要补什么证据 |
|--------|--------|--------|
| Traditional RAG | 文档能否被切分、召回与排序 | 服务是否健康、资料状态是否可追踪 |
| GraphRAG | 实体关系能否形成证据链 | 图写入 Neo4j 后如何只读观察与溯源 |
| Nano Brain | 知识产物能否被组织和检索 | 模块是否在统一入口下被正确调用 |
| Agent Gateway | 请求能否被编排、会话能否持续 | 身份、资源、动作与内部 Token（服务间调用的可信凭据）如何共同约束调用 |
| Web / Platform | 功能能否在页面上被操作 | 页面动作如何回到后台表、会话与源码规则 |

</div>

> **【常见误解】**：“我已经把仓库克隆到本地”只说明代码存在；“Docker 里容器是绿色”只说明服务进程达到健康条件；“页面按钮能点击”也只说明前端发出了动作。完整证据必须继续回答数据落在哪里、请求以谁的身份执行、规则在哪一道闸门允许或拒绝。

### 1.2 运行服务与组件

> **本节要解决什么**：建立服务职责地图，知道每一种运行状态应去哪里寻找证据。

&emsp;&emsp;Compose 不是把全部代码塞进一个“巨型容器”，而是把职责不同的服务连成一套系统。你可以把它理解为办公楼：Web 是前台入口，API 与 Gateway 是业务走廊，三条 RAG 是专门资料室，PostgreSQL 与 Neo4j 是不同类型的档案室，`migrate`（一次性执行建库、表结构升级和初始化的任务）则在开门前完成准备。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>运行单元与可观察证据</font></p>
<div class="center">

| 运行单元 | 负责什么 | 本节怎样验证 |
|--------|--------|--------|
| `web` | 浏览器访问入口 | 访问 Web 与一个具体功能区域 |
| `api` / `agent-gateway` | 接收业务请求、转发身份与模块调用 | 把 UI 动作回接到服务与规则 |
| `nano-brain` / `traditional-rag` / `graph-rag` | 三条独立检索与处理边界 | 本节确认服务边界，真实摄取另列待补证 |
| `postgres` | 业务、状态、会话与检索辅助数据 | DBeaver 连接确认与业务 `SELECT` |
| `neo4j` | GraphRAG 的实体、关系与来源线索 | Browser 中查看局部子图 |
| `migrate` | 建库、迁移和初始化 | 成功完成后退出是正常状态 |

</div>

&emsp;&emsp;表中最重要的边界是：默认情况下只有 Web 通过端口映射（把宿主机端口转发到容器内服务端口）提供给宿主机。这里的宿主机就是运行 Docker Desktop 的本机；数据库和内部服务不是“忘了开端口”，而是刻意留在 Compose 网络内。后面临时开放观察端口时，我们仍然是在看同一份数据卷，不是在启动第二套系统。

> **【判断边界】**：`migrate` 的目标是完成一次性工作，因此 `exited (0)` 是成功；常驻服务（需要持续运行以接收请求的服务）才应持续显示运行或健康状态。健康状态来自 healthcheck（容器定期执行的自检命令）。反例是 `migrate` 非零退出、或常驻服务反复重启：此时不能用 Web 是否偶尔可打开替代排查。

### 1.3 访问入口与连接边界

&emsp;&emsp;同一套系统里同时存在三种地址语义。第一种是默认产品入口，我们通过宿主机的 Web 地址进入系统；第二种是容器内部服务名，只供 Compose 网络中的服务相互调用；第三种是临时观察入口，只在本机回环地址上开放 PostgreSQL 与 Neo4j。它们不是同一地址的不同写法，而是三个不同的网络边界。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三种入口的职责边界</font></p>
<div class="center">

| 边界 | 谁使用 | 典型对象 | 本节怎样处理 |
|--------|--------|--------|--------|
| 默认产品入口 | 宿主机浏览器 | Web | 长期保留，用于真实功能操作 |
| Compose 内部网络 | 容器内服务 | API、Gateway、三条 RAG、数据库服务名 | 不向宿主机暴露，不使用容器 IP |
| 临时观察入口 | 本机 Neo4j Browser 与 DBeaver | Neo4j HTTP/Bolt、PostgreSQL | 观察时打开，全部 SQL 完成后关闭 |
| 业务授权边界 | 已认证用户与内部服务 | 资料、场景、会话、Agent 工具 | 由身份、归属、scope（资源可访问范围）、动作与 Token（服务间调用凭据）共同判断 |

</div>

&emsp;&emsp;可以把它们理解成办公楼的正门、楼内走廊和临时档案阅览口。正门能进入产品，不等于可以直达每个内部服务；阅览口能看数据库，也不等于可以绕过业务权限修改数据。后面我们只使用稳定的宿主机回环地址，不记录、不依赖容器 IP。

> **【判断边界】**：数据库观察口绑定在 `127.0.0.1`，只能说明它没有直接监听所有宿主机网卡；它不会把数据库账号变成只读账号。本课还会启用 DBeaver 连接保护、设置会话只读、只执行冻结的 `SELECT` 并限制返回字段，但当前 `postgres` bootstrap（初始化）账号仍不是专用只读角色。

### 1.4 PostgreSQL 与 Neo4j 的职责分层

&emsp;&emsp;GraphRAG 同时接触 PostgreSQL 与 Neo4j，最容易被误解成“同一份业务数据存了两遍”。真实职责不是简单复制：Neo4j 擅长表达实体和关系，PostgreSQL 负责业务对象、处理状态、向量与键值（KV）辅助数据、会话索引。它们共同支持一次请求，但回答的是不同问题。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>两类数据存储分别回答什么</font></p>
<div class="center">

| 存储 | 主要职责 | 适合回答的问题 | 本节的观察方式 |
|--------|--------|--------|--------|
| PostgreSQL | 用户、组织、资料、场景、处理状态、会话与检索辅助记录 | “谁拥有资源”“状态是否完成”“会话属于谁” | DBeaver 中的最小字段 `SELECT` |
| Neo4j | GraphRAG 实体、关系、类型与来源线索 | “谁与谁有关”“关系依据来自哪里” | Neo4j Browser 局部子图 |
| 应用规则 | 把身份、资源范围和当前动作组合成业务判定 | “当前用户为什么能读或不能管理” | 按源码规则编写的只读权限 SQL |

</div>

&emsp;&emsp;关系图不能替代业务表，数据库行也不能替代图关系；查询用户权限时，还必须把当前用户上下文与资源 owner、组织、团队和范围放进同一条规则中比较。第五章会把这条规则直接写成可观察的只读 SQL。

### 1.5 本节学习路径

&emsp;&emsp;本节不是按技术名词平铺，而是沿一次可复核的学习路径推进。每一站都留下一个结果，下一站只使用上一站已经确认的边界。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>课件包内源码与材料速查</font></p>
<div class="center">

| 主题 | 课件包相对路径 |
|--------|--------|
| 默认 Compose | `ff-companybrain/deploy/compose/compose.student.yml` |
| 本地镜像构建覆盖 | `ff-companybrain/deploy/compose/compose.build.yml` |
| 临时观察端口 | `ff-companybrain/deploy/compose/compose.dev-ports.yml` |
| 启动与复现说明 | `ff-companybrain/docs/deployment/student-start.md` |
| GraphRAG 双存储装配 | `ff-companybrain/modules/graph-rag/src/graph_rag/core/lightrag_service.py` |
| Neo4j workspace 边界 | `ff-companybrain/modules/graph-rag/src/graph_rag/core/neo4j_boundary.py` |
| 平台资源权限 | `ff-companybrain/packages/platform/src/platform-store.ts` |
| 会话私有边界 | `ff-companybrain/apps/agent-gateway/src/core/conversations.ts` |
| Agent 内部服务调用 | `ff-companybrain/apps/agent-gateway/src/agent/module-http-tools.ts` |

</div>

&emsp;&emsp;读到这里，我们还没有运行任何命令，但已经知道接下来每一步为什么存在、使用哪一种入口、应该留下什么证据。下面才开始部署。

## <center>第二章：Docker Compose 部署与本机数据库观察</center>

&emsp;&emsp;第一章解决了“系统由什么组成”；这一章开始让系统真正运行。我们先确认目录、Docker 与私有配置，再启动默认 Compose，并用迁移、常驻服务和 Web 三类证据判断结果。默认边界成立后，才临时开放 PostgreSQL 与 Neo4j 的本机观察端口。

&emsp;&emsp;这里要先拆掉两个误解。`migrate` 显示 `exited (0)` 不是服务挂了，而是一次性初始化正常完成；观察覆盖（在默认 Compose 配置之上追加少量观察端口的配置）也不是新启动一套数据库，而是给同一容器和同一数据卷增加回环映射。任何一步失败，都先记录失败信号，不通过重置数据卷来掩盖问题。

&emsp;&emsp;本章的系统命令全部在 Jupyter Notebook 的代码单元执行。每条命令以 `!` 开头，由 Notebook 调用 Shell。主线只保留“环境检查 → 私有配置 → 显式启动 → 状态判断 → 临时观察口”五步；镜像下载与完整日志属于首次构建或故障排查，不插入主流程。

> **【Notebook 执行边界】**：`!` 启动的是一次独立的 Shell 调用，因此不能依赖上一行单独执行的 `cd` 或 `export`。2.2 只定位课件包并保存 `ff-companybrain/` 的绝对路径；后续 Compose 命令显式使用该路径，不改变 Notebook 工作目录。重启内核后，先重新运行 2.2 的课件包定位单元。

> **【本章自学地图】**：安装并验证 Docker Desktop → 准备项目目录与私有配置 → 启动默认 Compose → 记录三类运行证据 → 确认默认端口 → 叠加临时观察端口 → 阅读覆盖文件解释边界。观察端口会保留到第五章全部 SQL 完成，再由 5.6 统一关闭。

### 2.1 Docker Desktop 与 Compose v2

> **本节要解决什么**：从一台尚未准备 Docker 的电脑开始，安装完整运行环境，并证明 Engine、CLI 与 Compose 都能使用。

&emsp;&emsp;本节使用 Docker Desktop 作为 macOS 与 Windows 的统一入口。它同时提供 Docker Engine（负责创建和运行容器的后台服务）、Docker CLI（在终端输入 `docker` 命令的命令行工具）和 Docker Compose，因此不需要再为本课单独寻找一个旧式的 `docker-compose` 安装包。安装页面与系统要求会随 Docker Desktop 更新，开始前请以官方文档为准：

- macOS：[Install Docker Desktop on Mac](https://docs.docker.com/desktop/setup/install/mac-install/)

- Windows：[Install Docker Desktop on Windows](https://docs.docker.com/desktop/setup/install/windows-install/)

- Compose 安装说明：[Overview of installing Docker Compose](https://docs.docker.com/compose/install/)

&emsp;&emsp;在 macOS 上，先确认芯片类型，再选择 Apple silicon 或 Intel 对应的安装包；完成安装后从“应用程序”启动 Docker Desktop。在 Windows 上，本节运行的是 Linux containers，优先使用 Docker Desktop 的 WSL 2（Windows 提供的 Linux 运行环境）路径；如果单位电脑受管理员策略、虚拟化设置或代理限制，应先解决这些宿主机前置，再进入项目部署。

&emsp;&emsp;安装完成不等于 Docker 守护进程已经可用。等待 Docker Desktop 显示运行后，在 Notebook 的一个代码单元执行下面两项检查：第一项确认客户端和服务端能通信；第二项确认本课使用的 `docker compose` 子命令可用。

> **【Notebook 单元执行原则】**：版本、路径和状态等快速检查可以放在同一个单元；镜像拉取、服务启动、端口覆盖、迁移确认和安全校验等耗时或会改变运行状态的操作各自独立成单元。这样每一步的输出、失败原因和重试范围都清楚可见。

In [ ]:
# 同时显示 Client 与 Server；只有 Client 通常表示守护进程尚未就绪
!docker version

# 本项目要求 Compose v2 子命令，命令形式是 docker compose
!docker compose version

> **【首次构建的网络前置】**：首次执行 2.3 的构建命令时，Docker 会自动拉取本机缺失的基础镜像；不需要逐个手动预检或拉取。如果网络不能直接访问 Docker Hub 或 GHCR，请先开启可用的代理/VPN网络，并确认 Docker Desktop 自己能够使用这条网络路径；浏览器或 Notebook 能联网并不代表 Docker Engine 可以拉取。Docker Desktop 的代理应在 Desktop 设置中配置，而不是写入本项目 `.env`。[Docker Desktop 代理设置](https://docs.docker.com/desktop/settings-and-maintenance/settings/)

&emsp;&emsp;本项目的 PostgreSQL 使用 Docker Hub 的 `pgvector/pgvector`；本地构建还会依赖 `node`、`oven/bun`、`neo4j` 与 GHCR 的 `astral-sh/uv` 等基础镜像。首次构建的下载进度会直接显示在 2.3 的输出中；若失败，先解决 Docker Desktop 的网络连通性，再重新执行同一个显式启动单元。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171624982.png" alt="Docker Desktop 中确认 pgvector PostgreSQL 基础镜像的本地缓存" width=95%></div>

&emsp;&emsp;上图只用于说明：可以在 Docker Desktop 的 **Images** 页面按完整镜像标签确认本机是否已有基础镜像。它不是要求逐张手工预拉镜像的操作步骤；实际构建仍以 2.3 的 Compose 命令为准，缺失镜像会由 Docker 自动拉取。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171624927.png" alt="Docker Desktop 中的企业知识中台容器组" width=82%></div>

&emsp;&emsp;上图展示 Docker Desktop 的容器列表区域。完成启动后，你将在这里看到 `ff-companybrain` 项目组、一次性 `migrate-1` 容器和各个常驻服务。它是界面位置参考；其中容器 ID、镜像标签、端口和状态均为一次本机快照，不应当作固定配置。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>安装验证的结果判读</font></p>
<div class="center">

| 现象 | 说明 | 下一步 |
|--------|--------|--------|
| `docker version` 同时出现 Client 与 Server | Engine 已经响应 | 继续检查 Compose |
| 只有 Client 或提示无法连接 daemon | Desktop 未启动、未就绪或当前 context 不正确 | 打开 Desktop，等待就绪后重试 |
| `docker compose version` 正常输出 | Compose v2 可用 | 进入项目目录准备 |
| 只有 `docker-compose` 可用 | 当前环境仍依赖旧入口 | 不继续本节，先按官方说明补齐 Compose v2 |
| 显式构建命令拉取基础镜像失败 | Docker Desktop 未能访问镜像仓库 | 先配置 Desktop 的代理/VPN网络，再重试同一启动单元 |

</div>

> **【不要混淆】**：Docker Desktop 是宿主机运行环境，Compose 文件是项目的服务编排说明，镜像是服务运行模板，容器是镜像的一次运行实例，volume（容器外持久保存数据的存储卷）才承载需要保留的数据。重新创建容器不等于自动删除 volume；反过来，执行带数据清理含义的命令也不能当作普通重启。

### 2.1.1 Compose 配置文件与合并关系

> **本节要解决什么**：理解 Compose v2 为什么需要同时读取多份 YAML，以及三份配置文件分别改变了系统的哪一层。

&emsp;&emsp;Compose v2 不是业务服务，也不是容器。Docker Engine（真正创建和运行容器的后台服务）负责运行服务；Compose v2 读取 YAML 编排说明，再告诉 Engine 要启动哪些服务、如何连接网络、挂载哪些数据卷和发布哪些端口。本项目使用 `docker compose`，并按命令中 `-f` 出现的顺序把多份 YAML 合成一份最终运行配置：后面的文件只补充或覆盖同名服务的对应字段，不会凭空启动第二套系统。

```text
compose.student.yml
        + compose.build.yml
        = 本地源码构建后的默认运行栈

compose.student.yml
        + compose.build.yml
        + compose.dev-ports.yml
        = 同一运行栈 + 本机 Neo4j / PostgreSQL 临时观察口
```

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三份 Compose YAML 的职责</font></p>
<div class="center">

| 文件 | 相对 `ff-companybrain/` 的位置 | 包含什么 | 在本节的作用 |
|--------|--------|--------|--------|
| `compose.student.yml` | `deploy/compose/compose.student.yml` | 全部服务、镜像、环境变量引用、依赖顺序、健康检查、数据卷、内部网络与默认 Web 端口 | 系统的基础运行定义 |
| `compose.build.yml` | `deploy/compose/compose.build.yml` | `web`、API、Gateway、三条 RAG、Neo4j、`migrate` 的本地 Dockerfile / build context | 将同一批服务改为从当前源码构建镜像 |
| `compose.dev-ports.yml` | `deploy/compose/compose.dev-ports.yml` | 仅 `postgres`、`neo4j` 的回环端口和 `dev_database_viewer` 网络 | 第三至第五章观察期间临时叠加 |

</div>

&emsp;&emsp;基础文件 `compose.student.yml` 定义的是完整运行栈：`postgres` 与 `neo4j` 先健康，`migrate`（一次性迁移任务）完成后，三条 RAG、API、Agent Gateway 才能启动，最后 Web 等 API 与 Gateway 健康。它还声明 PostgreSQL、Neo4j、平台文件和 RAG 工作目录的数据卷。默认情况下，只有 `web` 使用 `ports` 向宿主机发布 `3000`；数据库和内部服务使用 `expose` 留在 Compose 内部网络中，`expose` 不会让宿主机客户端直接连接。

&emsp;&emsp;`compose.build.yml` 不是第二份完整配置。它只为基础文件中的同名服务增加 `build`：例如 `web` 使用 `deploy/compose/Dockerfile.web`，Python RAG 服务使用 Python 服务 Dockerfile。合并后，服务仍然保留基础文件中的网络、数据卷、健康检查和环境变量，只是镜像可由当前项目源码构建。因此，本课本地启动默认叠加 `student + build`；如果没有这份构建覆盖，Compose 只能依赖基础文件中指定的预构建镜像。

&emsp;&emsp;`compose.dev-ports.yml` 也不是开发版业务栈。它只给两个数据存储追加一个用于桌面客户端的桥接网络，并从 `.env` 读取三个仅本机回环可达的观察端口：`POSTGRES_DEV_HOST_PORT`（默认 `15432`）、`NEO4J_HTTP_DEV_HOST_PORT`（默认 `17474`）、`NEO4J_BOLT_DEV_HOST_PORT`（默认 `17687`）。API、Web、Agent Gateway 和三条 RAG 都不加入这个观察网络；因此 DBeaver 和 Neo4j Browser 能观察同一份数据，却不会因此把内部业务 API 暴露到宿主机。观察入口会从第三章保留到第五章，全部数据库查询完成后再统一关闭。

> **【配置与私有值的边界】**：YAML 保存服务结构和变量名；实际密码、Token、模型地址等私有值由 `deploy/compose/.env` 提供。`${变量:?提示}` 表示变量缺失时 Compose 应停止并给出提示，不能用截图或代码单元打印 `.env` 内容来排查。

> **【最容易误解的边界】**：叠加 `compose.dev-ports.yml` 改变的是“宿主机从哪里观察容器”，不是“业务服务如何处理请求”，也不是“启动新的 PostgreSQL / Neo4j 数据库”。三份 YAML 合成后仍使用同一个 Compose 项目、同一组服务和同一批数据卷；第五章完成全部 SQL 后，会重新按不含观察覆盖的默认配置收敛端口边界。

### 2.2 课程目录与私有配置

> **本节要解决什么**：确认当前课件包位置，并准备不公开的 Compose `.env` 配置文件。

&emsp;&emsp;**先拆误解**：这一节不是在讲 Python 路径技巧，也不会启动 Docker。当前交付采用扁平课件包：本 Notebook 与 `ff-companybrain/` 位于同一目录。这里仅确认源码包的位置，并在私有 `.env` 不存在时从模板创建一次；第四章会直接在 DBeaver 中完成数据库观察。

```text
课件资料/
├── 企业级知识中台·第四节课：Docker Compose 部署、Neo4j 与权限边界.ipynb
└── ff-companybrain/
```

In [ ]:
# 当前 Notebook 目录应包含源码包
import os
import shutil
from pathlib import Path

COURSEWARE_DIR = Path.cwd().resolve()
FF_REPO = COURSEWARE_DIR / "ff-companybrain"
COMPOSE_DIR = FF_REPO / "deploy" / "compose"
COMPOSE_ENV_FILE = COMPOSE_DIR / ".env"
COMPOSE_ENV_TEMPLATE = COMPOSE_DIR / ".env.example"

# 只验证课件包位置；不改变 Notebook 工作目录
PACKAGE_READY = (
    FF_REPO.is_dir()
    and COMPOSE_ENV_TEMPLATE.is_file()
)
if not PACKAGE_READY:
    print("[SETUP_BLOCKED] 请从与 ff-companybrain/ 同级的课件资料目录启动 Notebook。")
elif not COMPOSE_ENV_FILE.exists():
    # 只在文件缺失时复制模板，并限制本机文件权限
    shutil.copy2(COMPOSE_ENV_TEMPLATE, COMPOSE_ENV_FILE)
    os.chmod(COMPOSE_ENV_FILE, 0o600)
    print("[SETUP_BLOCKED] 已创建私有 .env；请先填写全部 CHANGE_ME 项，再进入 2.3。")
else:
    # 只检查模板标记，不回显任何配置值
    env_text = COMPOSE_ENV_FILE.read_text(encoding="utf-8")
    env_values = {
        key: value
        for line in env_text.splitlines()
        if line and not line.lstrip().startswith("#") and "=" in line
        for key, value in [line.split("=", 1)]
    }
    if "CHANGE_ME" in env_text or "change-me-internal-token" in env_text:
        print("[SETUP_BLOCKED] 私有 .env 仍含模板值；请填写后重新运行本单元。")
    elif len(env_values.get("RAG_INTERNAL_TOKEN", "")) < 32:
        # 与源码启动护栏一致，只判断长度，不打印 Token
        print("[SETUP_BLOCKED] RAG_INTERNAL_TOKEN 至少需要 32 个字符。")
    else:
        print("[INFO] 私有 .env 已通过模板标记与内部 Token 长度检查。")

&emsp;&emsp;这一步只负责定位课件包、创建私有文件并检查是否残留模板标记；它不会回显 `.env` 内容。YAML 保存服务结构与变量名；密码、Token、模型地址和本机端口等私有值只保存在 `ff-companybrain/deploy/compose/.env`。可以把 2.2 理解为“找到机房并领取、检查配置表”，而不是“按下开机按钮”。

> **【判断边界】**：复制 `.env.example` 不等于配置已经可用。继续 2.3 前，必须在编辑器中替换全部 `CHANGE_ME`；Compose 能检查 YAML 中声明的缺失或空变量，但不能判断一个仍是模板文字的非空值是否是真实凭据。不要把 `.env` 内容粘进代码单元、截图或笔记。

### 2.3 显式构建与启动

> **本节要解决什么**：从本地源码构建并启动默认服务栈，同时明确本次实际合并的 Compose 文件。

&emsp;&emsp;**先拆误解**：`compose.build.yml` 不是“自动打开开发端口”的开关，它只为基础文件中的同名服务补充本地 Dockerfile 与构建上下文。本课默认启动显式合并 `compose.student.yml + compose.build.yml`，不加入 `compose.dev-ports.yml`；只有 2.6 才临时增加数据库观察入口。

&emsp;&emsp;现在启动默认边界。下面的命令从私有 `.env` 读取配置，按当前源码构建镜像并在后台启动服务。首次运行时，Docker 会自动下载缺失的基础镜像；下载和构建可能需要数分钟，请保持 Docker Desktop 运行。执行前应已完成 2.2，并在编辑器中替换全部 `CHANGE_ME`。

In [ ]:
# 显式叠加 student 与 build；默认不加载观察端口覆盖
!docker compose --project-name ff-companybrain --env-file "$FF_REPO/deploy/compose/.env" -f "$FF_REPO/deploy/compose/compose.student.yml" -f "$FF_REPO/deploy/compose/compose.build.yml" up --build --detach

> **【源码锚点】**：`ff-companybrain/deploy/compose/compose.student.yml` 定义完整服务、依赖、数据卷与默认 Web 端口；`compose.build.yml` 只补充本地构建配置。当前 `up.sh build` 还会额外加载 `compose.dev-ports.yml`，因此本课使用上面的显式两文件命令表达“默认启动”，不把脚本的真实行为写成另一种含义。

> **【观察端口会被收回】**：如果此前打开过 Neo4j Browser 或 SQL 观察口，重新执行这条显式两文件命令会把 PostgreSQL 与 Neo4j 收敛回默认边界，不再发布三个观察端口。Browser 页面可能暂时仍显示在浏览器中，但 Bolt 查询连接会失败；这不是密码错误。需要继续观察时，再执行 2.6 的三文件命令。

&emsp;&emsp;启动完成后，在 Docker Desktop 的 **Containers** 页面展开 `ff-companybrain` 项目组，依次确认三类信号：

1. `migrate-1` 显示 `Exited (0)`；

2. PostgreSQL、Neo4j、三条 RAG、API、Agent Gateway 和 Web 保持运行，且没有持续重启；

3. 在浏览器打开 `http://127.0.0.1:3000`，确认 Web 入口可访问。

&emsp;&emsp;这三类信号分别证明迁移任务、常驻服务与产品入口，不能相互替代。Web 可访问不证明三条 RAG 已完成真实摄取，也不证明外部模型可用；这些结论需要后续章节的单独证据。若启动失败，先保留 Docker Desktop 中失败服务的状态和本地日志，再修正 `.env` 或资源前置；不要使用会删除数据卷的 `reset.sh --yes`。

&emsp;&emsp;不要把“已经创建过表”理解成 `migrate-1` 从此永远不执行。每当 Compose 需要启动或重建这个容器时，它都会再次运行“检查并补齐”流程：已有表、索引和扩展不会重复创建；新加入的表、索引或字段会被补上；运行账号权限、默认管理员和启动后检查也会重新核对。完成后退出为 `Exited (0)`，正是一次性任务成功结束的信号。

&emsp;&emsp;这种特性叫**幂等**：同一操作重复执行，最终结果与执行一次相同。源码里的 `CREATE TABLE IF NOT EXISTS`、`CREATE INDEX IF NOT EXISTS` 和 `ALTER TABLE ... ADD COLUMN IF NOT EXISTS` 就是这个保证。可以把 `migrate-1` 想成开课前的教室检查：它每次都会核对桌椅、投影和名单；已经齐全的不会再添一套，缺少的新设备才会补上。只重启某个已经运行的业务容器通常不会触发它；再次执行 Compose 启动、重建或显式重建 `migrate` 时才会重新运行。

> **【源码锚点】**：在 `ff-companybrain/scripts/init-db.ts` 查看初始化顺序（数据库底座 → 应用迁移 → 权限 → 默认管理员 → postcheck）；在 `packages/identity/src/migrations.ts` 或 `packages/platform/src/migrations.ts` 搜索 `IF NOT EXISTS`，可核对表、索引和字段的幂等写法。

> **【常见故障】**：如果迁移日志明确报缺少配置，先补齐私有 `.env` 再重启；如果 Docker 资源不足，先调整 Docker Desktop 资源后重新观察状态；如果 Web 返回失败但服务仍在运行，继续查看 `web` 与其依赖服务日志。不要运行 `reset.sh --yes` 来处理普通启动错误，因为它会删除本项目数据卷。

### 2.4 启动证据记录

> **本节要解决什么**：留下不含密钥和业务正文的最小证据，区分本次启动与旧容器残留。

&emsp;&emsp;服务状态是会变化的，因此不要只凭记忆说“昨天能跑”。为这次启动建立一条最小记录：启动命令是否返回非零、迁移是否完成、常驻服务是否稳定、Web 是否响应、观察端口是否尚未打开。记录不需要包含容器 ID、镜像地址或环境变量；它只需要让你下次能区分“系统曾经启动”和“这次启动成功”。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Compose 启动证据的最小记录</font></p>
<div class="center">

| 证据项 | 你要记录什么 | 什么不应写入记录 |
|--------|--------|--------|
| 私有配置 | 显式 Compose 命令是否接受 `.env` | `.env` 全文、密钥和值 |
| migrate | 正常完成或失败日志的类别 | 完整数据库连接串 |
| 常驻服务 | 服务名、运行/健康状态、重启异常 | 容器 ID、私有镜像仓库 |
| Web | 浏览器或 HTTP 响应是否可达 | 登录账号、页面正文 |
| 端口边界 | 默认只有 Web，观察口是否未启用 | 外网 IP、内部网络细节 |

</div>

&emsp;&emsp;把这张记录理解成机场起飞前的检查单，而不是运行报告的装饰。缺少任一项时，后面看到的数据库或 UI 现象都可能来自残留容器、旧数据卷或另一次启动，无法作为本次复现的证据。

### 2.5 默认端口边界

> **本节要解决什么**：在增加观察端口前先保存默认基线，明确数据库未对宿主机开放。

&emsp;&emsp;在默认 Compose 中，浏览器应只通过 Web 入口访问系统。启动完成后，在 Docker Desktop 的 **Port(s)** 列确认 `web` 出现宿主机产品端口；PostgreSQL 与 Neo4j 不应显示 `15432`、`17474` 或 `17687`。若命令行偶尔显示 `3002/tcp`、`8100/tcp` 这类内部端口，不要把它当成宿主机映射；真正发布到宿主机的格式会包含地址与箭头，例如 `0.0.0.0:3000->3000/tcp` 或 `127.0.0.1:15432->5432/tcp`。这一判断建立在 2.1.1 的基础配置上：`compose.student.yml` 只发布 Web，数据库使用 `expose` 留在 Compose 内部网络。

&emsp;&emsp;如果数据库端口已经出现，先不要继续 UI 或 SQL 操作。它说明当前项目仍保留观察覆盖；应先重新执行 2.3 的显式 `compose.student.yml + compose.build.yml` 命令，或在第五章结束时按 5.6 收敛端口。项目的 `verify-stack.sh` 是完整的集成校验：除端口检查外，还会等待服务、探测受保护 Tool 接口并执行一次常规服务重启；它不适合放在本章的启动主线中。

> **【源码锚点】**：`ff-companybrain/deploy/compose/compose.student.yml` 仅为 `web` 声明宿主机 `ports`；`ff-companybrain/deploy/compose/verify-stack.sh` 的 `wait_for_full_stack`、`probe_protected_tool_http` 与 `restart` 流程说明它并非单纯端口查看命令。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>本节使用的入口边界</font></p>
<div class="center">

| 使用场景 | 入口 | 用途与边界 |
|--------|--------|--------|
| 默认产品入口 | `http://127.0.0.1:3000` | 打开真实 Web 功能；不是数据库客户端 |
| 临时 PostgreSQL 观察 | `127.0.0.1:${POSTGRES_DEV_HOST_PORT}`（默认 `15432`） | DBeaver 业务查询入口；只绑定本机回环地址，端口本身不保证只读 |
| 临时 Neo4j Browser | `http://127.0.0.1:${NEO4J_HTTP_DEV_HOST_PORT}/browser/`（默认 `17474`） | 图关系观察与 Cypher 编辑器 |
| 临时 Neo4j Bolt | `127.0.0.1:${NEO4J_BOLT_DEV_HOST_PORT}`（默认 `17687`） | Browser/驱动使用的图数据库连接协议 |

</div>

&emsp;&emsp;不要把容器内部主机名或容器 IP 复制到宿主机工具里。宿主机使用稳定的回环地址；容器之间才通过 Compose 网络和服务名互相访问。这是“网络位置不同，地址语义不同”的边界例。

### 2.6 本机数据库观察端口

> **本节要解决什么**：只在需要观察时，为同一套 PostgreSQL 与 Neo4j 临时增加回环入口。

&emsp;&emsp;**先拆误解**：这不是“开发版业务栈”，也不是新启动一套 PostgreSQL 或 Neo4j。它在 2.3 的默认两文件组合上额外叠加 `compose.dev-ports.yml`，只为同名数据库容器增加本机观察入口；服务、数据卷和业务处理逻辑仍属于同一个 Compose 项目。

In [ ]:
# 显式加入观察覆盖；三个数据库入口都只绑定 127.0.0.1
!docker compose --project-name ff-companybrain --env-file "$FF_REPO/deploy/compose/.env" -f "$FF_REPO/deploy/compose/compose.student.yml" -f "$FF_REPO/deploy/compose/compose.build.yml" -f "$FF_REPO/deploy/compose/compose.dev-ports.yml" up --detach

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>`.env` 中的本机端口配置</font></p>
<div class="center">

| `.env` 变量 | 模板默认值 | 用途 | 何时生效 |
|--------|--------|--------|--------|
| `WEB_HOST_PORT` | `3000` | Web 产品入口 | 默认启动与观察启动 |
| `POSTGRES_DEV_HOST_PORT` | `15432` | PostgreSQL 本机观察入口 | 仅叠加 `compose.dev-ports.yml` 后 |
| `NEO4J_HTTP_DEV_HOST_PORT` | `17474` | Neo4j Browser HTTP 页面 | 仅叠加 `compose.dev-ports.yml` 后 |
| `NEO4J_BOLT_DEV_HOST_PORT` | `17687` | Neo4j Browser / 驱动查询连接 | 仅叠加 `compose.dev-ports.yml` 后 |

</div>

&emsp;&emsp;成功时，在 Docker Desktop 的 **Port(s)** 列会看到 PostgreSQL、Neo4j HTTP 与 Bolt 的回环映射；具体端口以私有 `.env` 为准，表中的数字只是模板默认值。如果端口被占用，先停止占用同一端口的本地服务，或修改 `.env` 中对应变量后重新运行本节命令。不要把绑定地址改成 `0.0.0.0` 来“快速解决”冲突，那会把仅本机可见的观察窗变成网络暴露入口。

> **【源码锚点与反例】**：`ff-companybrain/deploy/compose/compose.dev-ports.yml` 只修改 `postgres` 与 `neo4j`，三个映射均固定绑定 `127.0.0.1`。Browser 能打开 `17474` 只证明 HTTP 观察入口可达，不能证明 Bolt、图数据、GraphRAG 摄取或 Agent 调用已经成功。

### 2.7 观察网络边界

> **本节要解决什么**：从覆盖文件验证端口对象和绑定地址，不凭 Docker Desktop 的界面猜测网络边界。

&emsp;&emsp;观察覆盖的配置关系已经在 2.1.1 给出：它只给 `postgres` 加 `127.0.0.1:${POSTGRES_DEV_HOST_PORT}:5432`，给 `neo4j` 加由 `NEO4J_HTTP_DEV_HOST_PORT` 与 `NEO4J_BOLT_DEV_HOST_PORT` 决定的回环映射；Web、API、Gateway 与三条 RAG 并没有因此加入一个对宿主机开放的新网络。

&emsp;&emsp;这回答了一个常见边界问题：为什么 Browser 能连接 Neo4j，而 API 或 RAG 的宿主机端口仍然不存在？因为观察覆盖只额外连接两个数据存储，业务服务继续使用内部网络。把数据库观察口误当成“所有内部 API 都应该有端口”，会把开发观察与业务架构混为一谈。

> **【反例】**：浏览器能打开 `17474` 并不说明 GraphRAG 摄取成功；它只说明 Neo4j Browser 的观察入口可达。图数据是否存在、文档状态是否 ready、Agent 能否使用结果，分别需要第三、四、五章的不同证据。

### 2.8 Windows 与本地服务冲突处理

> **本节要解决什么**：明确 Windows 可复现的运行入口，以及本机同名数据库为什么不能替代 Compose 服务。

&emsp;&emsp;如果你使用 Windows，也可以运行这套 Docker 教学栈。本章已经直接展示 `docker compose` 命令，不依赖 `up.sh` 作为默认启动入口；当前可复现路径仍是 Docker Desktop 的 Linux containers/WSL2 模式，并从 WSL 或 Git Bash 启动 Notebook 后执行 `!` 命令。容器带走了 Bun、Python、uv（Python 依赖与虚拟环境工具）、PostgreSQL 与 Neo4j 等服务运行时，却不会替你提供私有 `.env`、模型 Key、虚拟化前置或可用网络。

&emsp;&emsp;这不是“Windows 不支持”的结论，而是运行入口的边界说明。Windows 冷启动仍属于待补证项：当你在 Windows 实机完成从解压源码、配置私有环境、启动、访问 Web 到最小链路验证的全过程后，才能把该结果填入证据卡。

> **【自查问题】**：如果我在宿主机本地安装了 PostgreSQL 或 Neo4j，会不会代替 Docker 中的服务？不会。本节使用的服务、数据卷和观察端口都属于 Compose 项目；本地另装的同名软件反而可能占用端口并制造混淆。

### 2.9 部署状态判定

> **本节要解决什么**：把“容器看起来启动了”进一步判断为“可以进入下一章”或“必须先修复前置”。

&emsp;&emsp;完成第二章后，用下面五张状态卡做一次判断。先写“可以继续 / 不可以继续”，再写缺少的证据。不要把“我能打开某个页面”作为所有状态的通用答案。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>部署状态判断练习</font></p>
<div class="center">

| 状态卡 | 你的判断 | 正确的证据边界 |
|--------|--------|--------|
| `migrate exited (0)`，八个常驻服务稳定，Web 可访问 | 可以进入观察准备 | 只证明启动级基线，不证明真实摄取 |
| Web 可访问，但 `migrate exited (1)` | 不继续 | 入口偶尔响应不能覆盖迁移失败 |
| 所有服务 healthy，但模型 Key 尚未配置 | 可以学习本地观察，不能宣称 Agent/RAG 端到端通过 | 健康检查不验证外部模型结果 |
| PostgreSQL 映射到 `0.0.0.0` | 不继续 | 观察入口已经超出本机回环边界 |
| Browser 页面可打开，但 Bolt 无法连接 | 先修复观察入口 | HTTP 页面可达与图查询通道可达是两件事 |

</div>

&emsp;&emsp;你的章末记录至少应包含：Docker 与 Compose 是否可用、配置检查结果、migrate 状态、常驻服务状态、Web 入口、默认端口边界、观察端口边界。缺少其中一项时，写明“缺少什么”，不要写“部署基本完成”。

## <center>第三章：Neo4j 图数据只读观察</center>

&emsp;&emsp;GraphRAG 同时使用两类存储：Neo4j 保存实体与关系，PostgreSQL 保存 KV、向量、文档状态和业务辅助记录。它们不是可互换的两个数据库，而是同一条检索链路中回答不同问题的两个位置。

&emsp;&emsp;Neo4j Browser 也不是业务管理页面。它是开发者观察台：适合回答“谁和谁有关、这条边是什么意思、证据来自哪里”，不适合浏览或修改全部业务资料。把 Browser 的图形效果误当成业务权限结果，是本节要避免的边界错误。

&emsp;&emsp;这一章依次完成连接、只读模式、规模统计、局部子图和来源字段核对。源码锚点是 `ff-companybrain/modules/graph-rag/src/graph_rag/core/lightrag_service.py`：其中将 `PGKVStorage`（键值辅助数据）、`PGVectorStorage`（向量检索数据）、`PGDocStatusStorage`（文档处理状态）分配给 PostgreSQL，将 `Neo4JStorage`（图关系存储）分配给 Neo4j。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171624947.png" alt="PostgreSQL 与 Neo4j 在三条 RAG 链路中的职责分工" width=78%></div>

&emsp;&emsp;这张图说明两类数据库在 GraphRAG 链路中的分工：PostgreSQL 保存可检索、可恢复的业务与处理状态；Neo4j 保存实体和关系；业务授权仍由 API / store 规则判断，不能仅凭某一张表或某一张图推导访问权限。

> **【本章自学地图】**：连接 Browser → 切换只读模式 → 先看规模再看局部关系 → 区分 workspace 与业务类型 → 使用 Graph/Table/RAW → 用来源字段完成溯源。整章只观察，不执行创建、修改或删除。

### 3.1 Neo4j Browser 连接地址

> **本节要解决什么**：区分 Browser 的 HTTP 页面地址与 Bolt 查询地址，并能判断连接失败发生在哪一层。

&emsp;&emsp;先查看 `.env` 中的 `NEO4J_HTTP_DEV_HOST_PORT` 与 `NEO4J_BOLT_DEV_HOST_PORT`：模板默认值时，打开 `http://127.0.0.1:17474/browser/`，并在 Browser 连接界面填入 `bolt://127.0.0.1:17687`；如果你改过端口，就用 `.env` 中的当前值替换这两个默认数字。数据库选择 `neo4j`；用户名与密码只从你自己的私有 Compose 环境读取，绝不写入命令、截图或笔记。HTTP 地址负责打开界面，Bolt（Neo4j 客户端执行图查询所用的连接协议）地址负责图查询通信，它们连接同一个 Neo4j 容器。

&emsp;&emsp;连接失败时先判断是哪一层失败：Browser 页面打不开，先检查观察覆盖、`.env` 的 `NEO4J_HTTP_DEV_HOST_PORT` 与 `neo4j` 服务状态；页面能开但 Bolt 连接失败，检查 `.env` 的 `NEO4J_BOLT_DEV_HOST_PORT`、回环映射和私有凭据；连接后没有可读数据，则记录为图数据或摄取前置尚未完成，而不是虚构查询结果。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171634684.png" alt="Neo4j Browser 的查询与结果区域" width=82%></div>

&emsp;&emsp;上图是一次真实的 Browser 只读观察：左侧列出节点数、关系类型和可用属性键；中间 Table 显示规模查询结果。节点数和关系数会随摄取结果变化，图中的 `gsrc_*` 只是当次资料源的内部标识，截图或课堂记录中不应把它当作业务名称或公开内容。

### 3.2 Neo4j Browser 只读会话

> **本节要解决什么**：先限制当前会话的操作意图，再开始观察图数据，避免把可连接误解为可随意修改。

&emsp;&emsp;在 Browser 编辑器执行第一条 Cypher（Neo4j 的图查询语言）。它不读取业务正文，也不修改图；作用是降低误操作风险，并提醒我们本节只做观察。运行后编辑器应接受只读会话设置；如果 Browser 报语法或连接错误，回到上一节排查连接，而不是尝试写入语句。

```cypher
// 将当前 Browser 会话切为只读观察模式
:access-mode read
```

&emsp;&emsp;只读模式类似档案室的阅览证：它限制当前操作意图，但不等于你可以展示所有资料。字段选择、截图脱敏和业务权限仍然由更高层的规则决定。

### 3.3 图数据规模与局部查询

> **本节要解决什么**：先用统计确认图中是否有数据，再用有限子图理解关系，避免一次加载全图。

&emsp;&emsp;先运行规模统计并切到 Table 视图。它给出当前快照的节点和关系数量，不应该被解释为系统容量上限；图中数据会随摄取、删除和资料源变化而改变。

```cypher
// 统计当前图快照；标量结果适合在 Table 视图查看
MATCH (n)
OPTIONAL MATCH ()-[r]->()
RETURN count(DISTINCT n) AS 节点数,
       count(DISTINCT r) AS 关系数;
```

&emsp;&emsp;如果统计结果是零，先检查当前资料是否已经完成 GraphRAG 摄取；这不是让你改写图的信号。你可以把零结果记录为 `SETUP_BLOCKED：缺少已摄取的脱敏资料源`，并继续阅读查询与字段含义。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171628919.png" alt="Neo4j Browser 中的图谱规模统计结果" width=88%></div>

&emsp;&emsp;上图是课堂教学 fixture 的一次真实 Browser 截图：Table 视图返回了节点数和关系数，左侧同时列出 workspace label、关系类型 `DIRECTED` 与可用属性键。图中的 `37`、`34` 和 `gsrc_*` 都是当次快照，不是本节查询的固定预期；本节要复现的是“查询得到计数，再用左侧结构信息解释图的组成”。

&emsp;&emsp;接着查询一个受限局部子图，再切换到 Graph 视图。`LIMIT` 是边界控制，不是性能装饰：它让你先看得懂一小块关系，而不是把整个图加载到浏览器画布。

```cypher
// 只返回有限关系，避免把整个图加载进浏览器
MATCH (a)-[r:DIRECTED]->(b)
RETURN a, r, b
LIMIT 50;
```

&emsp;&emsp;点击一个节点和一条关系，按四个问题记录：起点是谁、关系说明是什么、终点是谁、证据来自哪里。拖动节点只改变画布布局，不会写入 Neo4j；这是“可视化操作”和“数据库写入”必须区分的反例。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171624898.png" alt="Neo4j Browser 的局部公司知识关系图" width=84%></div>

&emsp;&emsp;上图展示同类局部关系查询在 Graph 视图中的效果：中心实体、协作方、负责人和资料节点由 `DIRECTED` 边连接。它是“雾桥计划”教学 fixture 的静态示例，用于认识 Graph 视图与关系方向；你自己的图中实体名称、节点数量和 workspace 都可以不同。

### 3.4 workspace 字段的业务边界

> **本节要解决什么**：分清资料源隔离标识与实体业务类型，不依据标签名称猜测业务含义。

&emsp;&emsp;资料源 workspace（按资料源隔离的图数据空间）用来隔离不同资料的图空间，`entity_type` 才描述人物、方法、内容或概念等业务语义。可以把 workspace 想成资料室编号，把 `entity_type` 想成档案分类；编号相同不代表业务类型相同，分类相同也不代表来自同一资料室。

```cypher
// 按实体业务类型统计，不把 workspace label 当作业务分类
MATCH (n)
RETURN n.entity_type AS 类型,
       count(*) AS 数量
ORDER BY 数量 DESC, 类型;

// 只读取节点的定位与来源线索，不读取原始文档正文
MATCH (n)
RETURN n.entity_id AS 实体,
       n.entity_type AS 类型,
       n.file_path AS 来源文件,
       n.source_id AS 来源片段
ORDER BY 类型, 实体;
```

&emsp;&emsp;如果某个属性为空，不要补写业务解释；不同摄取批次的字段完整度可能不同。你在这一章应带走的结论是“图关系存在哪里、如何只读观察、如何区分技术隔离与业务语义”，而不是某一份资料的具体内容。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171627906.png" alt="Neo4j Browser 中按 entity_type 统计节点类型" width=86%></div>

&emsp;&emsp;这张课堂实拍图对应上方的类型统计思路：`organization`、`person`、产品、客户、供应商等是节点的 `entity_type`，而左侧的 `gsrc_*` 是资料源隔离 label。截图中的类别与数量会随摄取资料变化，但“业务类型看 `entity_type`，隔离范围看 workspace”的判断规则不变。

> **【可复核锚点】**：在 `ff-companybrain/modules/graph-rag/src/graph_rag/core/lightrag_service.py` 搜索 `graph_storage="Neo4JStorage"` 与三个 `PG...Storage`；在 `neo4j_boundary.py` 搜索 `NEO4J_WORKSPACE is forbidden`。前者说明存储职责，后者说明资料源隔离不能被全局覆盖替代。

### 3.5 Neo4j Browser 界面区域

> **本节要解决什么**：让 Graph、Table、RAW 与左侧结构信息各自回答合适的问题，形成可重复的 UI 操作路径。

&emsp;&emsp;Browser 的同一份查询结果可以用不同视图理解。Graph 视图用于理解局部结构，Table 视图用于核对属性与计数，RAW（原始返回结构）视图主要用于连接或查询排障，左侧 Database information 用于识别标签、关系类型和属性键。它们不是四个不同的数据源，而是同一结果的不同观察方式。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Neo4j Browser 的问题与视图对应关系</font></p>
<div class="center">

| 你要回答的问题 | 优先区域 | 操作 | 不应据此推出什么 |
|--------|--------|--------|--------|
| 当前能否连接到图数据库 | 顶部连接信息与编辑器 | 核对本机 Bolt 地址和数据库名 | 不代表图中已有业务资料 |
| 当前图有多少节点和关系 | Table 视图 | 运行计数查询 | 不代表系统容量上限 |
| 某个实体如何关联 | Graph 视图 | 点击节点、边并整理局部图 | 拖动图形不会修改数据 |
| 一条边的实际含义 | Table 或属性面板 | 查看 description、weight、source 线索 | `DIRECTED` 名称本身不等于业务含义 |
| 某个字段为何为空 | RAW 与服务日志 | 看返回结构和错误提示 | 不能凭空补全来源字段 |

</div>

&emsp;&emsp;现在按一条固定的 UI 操作链做一次观察：在编辑器运行局部图查询；切换 Graph；点击中心节点；点击一条关系；切回 Table 核对字段；最后把节点、关系、属性、来源四项写入你的证据卡。每一步都只读，不在 Browser 中创建、删除或修改节点。

&emsp;&emsp;如果 Graph 视图没有连线，先确认查询是否 `RETURN a, r, b`，而不是只返回了名称或计数；如果结果过多，缩小查询并保留 `LIMIT`；如果属性面板显示敏感来源，停止截图并改用脱敏 fixture。Browser 的便利不改变数据最小化原则。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171628987.png" alt="Neo4j Browser 中的两跳供应链关系图" width=84%></div>

&emsp;&emsp;上图是课堂补充用的两跳业务关系图：从“本公司”出发，按一到两跳看到客户、供应商、产品与对接人。它只用来理解“沿关系继续查”的图查询价值，不应与本节 P/T/C 权限 fixture 混为同一份资料，也不要求你的环境返回相同节点数。

> **【认知锚点】**：Neo4j Browser 像一张可拖动的关系地图。地图告诉你道路如何相连，却不能替代道路通行证；业务授权、资料正文与服务调用仍在图外的边界中判断。

### 3.6 关系来源字段溯源

> **本节要解决什么**：从“看见一条连线”继续追到关系说明与来源线索，建立图谱可解释性的最小证据。

&emsp;&emsp;一张图只有连线时，容易变成“看起来很合理”的展示。GraphRAG 更重要的部分是能回到来源线索：节点或关系上的 `source_id`（来源片段标识）、`file_path`、`description`、`weight` 让你知道一条关系是怎样被解释和追踪的。这里仍然只读取元数据，不展开原始文本。

```cypher
// 以表格形式读取关系语义与来源线索，便于逐行核对
MATCH (a)-[r:DIRECTED]->(b)
RETURN a.entity_id AS 起点,
       r.description AS 关系含义,
       b.entity_id AS 终点,
       r.weight AS 权重,
       r.source_id AS 来源片段
ORDER BY 起点, 终点
LIMIT 50;
```

&emsp;&emsp;结果应在 Table 视图中阅读。`DIRECTED` 表示图存储使用的通用有向关系类型；真正的业务解释主要来自关系说明与来源线索。若 `source_id` 或 `file_path` 含敏感标识，停止截图，只记录“来源字段存在且已脱敏处理”的事实。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171624917.png" alt="Neo4j Browser 中的公司知识关系表" width=90%></div>

&emsp;&emsp;上图是课堂教学 fixture 的关系 Table 视图。重点不是记住“雾桥计划”这个样例名，而是按列读出：`source_entity` 是关系起点，`relation_keywords` 给出业务关系词，`target_entity` 是终点，`relation_description` 是可解释说明。图上的 `A6B5` 是 fixture 标识，不是 Neo4j 概念或业务字段规范。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>一次局部图观察应留下的四项记录</font></p>
<div class="center">

| 记录项 | 你要写什么 | 常见误读 |
|--------|--------|--------|
| 节点 | 一个脱敏 entity_id 与 entity_type | 把 workspace label 当成业务类型 |
| 关系 | 起点、终点、description 的最小解释 | 只看 `DIRECTED` 就下业务结论 |
| 属性 | 与问题相关的一个字段，例如 weight | 把缺失字段补成猜测 |
| 来源 | source_id 或 file_path 是否存在、是否脱敏 | 把来源线索当作可以公开的正文 |

</div>

&emsp;&emsp;如果你的图尚未具备可脱敏来源，就把本小节标为待补证。图谱可解释性不是要求你展示原始资料，而是要求你能说明“关系不是凭空出现的”，并保留合规的复核入口。

### 3.7 图数据与业务状态的分层排查

> **本节要解决什么**：面对“图里有、页面没有”或“状态 ready、图为空”等现象，先判断该查 Neo4j、PostgreSQL 还是应用权限。

&emsp;&emsp;下面四个现象故意不提供唯一的“修复命令”。你要先选择观察层，再说明还缺哪一类证据。这个练习的目标是阻止我们一看到异常就直接修改数据库。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>图与状态的分层诊断</font></p>
<div class="center">

| 现象 | 第一观察层 | 下一份证据 | 不应立即做什么 |
|--------|--------|--------|--------|
| Neo4j 有节点，GraphRAG source 状态缺失 | PostgreSQL / 受控模块接口 | source、document、delete_state 摘要 | 在 Neo4j 手工补业务状态 |
| source 显示 ready（资料源文档已完成处理），但局部图查询为空 | Neo4j 与 GraphRAG 服务日志 | workspace、关系类型、摄取批次 | 直接改成不带 workspace 的全局查询 |
| Browser 能看到关系，Bob 页面仍被拒绝 | 应用权限链 | 当前身份、资源范围、动作规则 | 认为“数据库里有数据就应该可见” |
| Table 里只有 `DIRECTED`，没有 description/source | 数据质量与摄取证据 | 关系属性、来源字段、摄取日志 | 依据边类型编造业务含义 |

</div>

&emsp;&emsp;完成练习后，你应能用一句话区分三种问题：图结构问题看 Neo4j，资料与会话状态看 PostgreSQL，当前用户能否执行动作看应用权限链。真实故障可能跨层，但排查必须从证据最接近的一层开始。

> **下一站**：Neo4j 已经回答“关系如何连接”；第四章转向 DBeaver，继续回答“身份是谁、资源属于谁、资料进入了哪条 RAG 链路”。

## <center>第四章：用 DBeaver 观察 PostgreSQL 业务数据</center>

&emsp;&emsp;Neo4j Browser 已经让我们看到实体与关系；本章转向 PostgreSQL，用 DBeaver 观察身份、场景、资料和索引对象分别存在哪里。本章所有数据库操作都在 DBeaver 中完成。

&emsp;&emsp;本章依次完成两件事：先建立两个数据库连接并确认当前会话的只读状态，再执行四组冻结的 `SELECT`，把“身份—资源—资料—索引”串成一条可复核的数据链。第五章会继续沿同一个观察入口计算 `can_read`、`can_manage` 和真实可读数据，因此本章结束时暂不关闭数据库端口。

&emsp;&emsp;**先拆误解**：DBeaver 的“连接成功”、回环端口 `127.0.0.1` 和客户端的 Read-only connection（只读连接）是三层不同证据。连接成功只说明网络和凭证可用；回环绑定只限制本机可达范围；DBeaver 的只读设置与 PostgreSQL 会话只读可以降低误操作风险，但本课使用的 bootstrap 账号（模板默认 `postgres`）不是专用只读角色，也不是 Web 中的 `admin` 用户。

> **【本章自学地图】**：建立 DBeaver 连接 → 启用客户端只读保护 → 设置并确认 PostgreSQL 会话只读 → 执行四组最小字段 `SELECT` → 判断当前结果能证明什么 → 保持观察入口进入第五章。

### 4.1 DBeaver 连接 PostgreSQL

> **本节要解决什么**：使用第二章已经开放的本机观察端口，为身份库和平台核心库建立 DBeaver 连接。

&emsp;&emsp;先确认已经完成 2.6，Docker Desktop 中的 PostgreSQL 映射应为 `127.0.0.1:${POSTGRES_DEV_HOST_PORT}->5432`。然后在 DBeaver 中新建 PostgreSQL 连接。主机固定为 `127.0.0.1`；端口读取私有 `.env` 中的 `POSTGRES_DEV_HOST_PORT`，模板默认 `15432`；用户名和密码分别读取 `POSTGRES_BOOTSTRAP_USER`、`POSTGRES_BOOTSTRAP_PASSWORD`。不要把密码、完整连接串或 `.env` 正文复制到课件、截图和 SQL 编辑器中。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>DBeaver 连接参数</font></p>
<div class="center">

| 配置项 | 本课填写方式 | 判断边界 |
|--------|--------|--------|
| Host | `127.0.0.1` | 宿主机连接不使用容器名或容器 IP |
| Port | `.env` 的 `POSTGRES_DEV_HOST_PORT`，模板默认 `15432` | 这是宿主机端口，不是容器内 `5432` |
| Database | 分别建立 `platform_identity_db` 与 `platform_core_db` 连接 | 切换数据库后需要重新确认会话 |
| Username | `.env` 的 `POSTGRES_BOOTSTRAP_USER` | 当前模板默认 `postgres`，不是业务用户 |
| Password | `.env` 的 `POSTGRES_BOOTSTRAP_PASSWORD` | 只在 DBeaver 私有连接中填写，不进入截图 |

</div>

&emsp;&emsp;在连接设置的 **Security** 中启用 **Read-only connection**，并限制数据编辑与结构编辑。这个开关能限制通过当前 DBeaver 连接进行修改，但它不等于项目已经创建了数据库权限层的只读账号。设置入口和边界以 [DBeaver 官方连接安全文档](https://dbeaver.com/docs/dbeaver/Managing-security-restrictions-for-database-connection/) 为准。

&emsp;&emsp;点击 **Test Connection** 只验证网络、驱动和凭证是否可用。测试成功后分别保存两个连接；若连接失败，先检查 2.6 的观察覆盖、私有 `.env` 和 PostgreSQL 容器状态，不要改用公开监听地址，也不要把凭证写进命令行。

### 4.2 会话只读确认

> **本节要解决什么**：在每个 DBeaver 数据库连接中设置并确认当前 PostgreSQL 会话默认只读，再执行后续业务查询。

&emsp;&emsp;**先拆误解**：勾选 DBeaver 的只读连接属于客户端保护；`transaction_read_only=on` 才描述 PostgreSQL 当前事务的只读状态。两者都不能把 `postgres` bootstrap 账号永久变成专用只读角色。为避免每次连接后忘记设置，在两个教学连接的 **Connection Initialization / Initialization** 中，把下面语句保存为每次连接时执行的 bootstrap SQL；不同 DBeaver 版本的菜单文字可能略有差异，以 [DBeaver Connection Initialization 官方文档](https://dbeaver.com/docs/dbeaver/Configure-Connection-Initialization-Settings/) 为准。

```sql
-- 每次建立连接时，让后续开启的事务默认只读
SET default_transaction_read_only = on;
```

&emsp;&emsp;保存后断开并重新连接，再在 SQL 编辑器执行下面的确认查询。不要只看工具栏的 auto-commit（自动提交）或 manual-commit（手动提交）：它们决定语句何时提交，不等于 PostgreSQL 已经拒绝写入。

```sql
SELECT
  current_database() AS database_name,                            -- 当前连接的目标数据库
  current_user AS database_user,                                 -- DBeaver 使用的数据库账号
  inet_server_port() AS server_port,                              -- PostgreSQL 容器内服务端口
  current_setting('default_transaction_read_only') AS default_read_only, -- 后续事务默认值
  current_setting('transaction_read_only') AS transaction_read_only;     -- 当前事务只读状态
```

&emsp;&emsp;继续查询前，先确认 `database_name` 与当前任务一致，`default_read_only` 和 `transaction_read_only` 都为 `on`。若任一值不是 `on`，停止业务查询，检查初始化设置并重新连接。`server_port` 通常返回容器内的 `5432`；DBeaver 使用的宿主机入口仍是 `.env` 中的 `POSTGRES_DEV_HOST_PORT`。PostgreSQL 官方文档说明，`default_transaction_read_only` 控制新事务的默认只读状态，而 `transaction_read_only` 反映当前事务状态；两者不能互相替代，详见 [PostgreSQL Client Connection Defaults](https://www.postgresql.org/docs/current/runtime-config-client.html)。

> **【安全边界】**：当前项目没有为课堂观察创建专用 readonly/observer（只读观察）角色，DBeaver 使用的是初始化账号。会话只读与客户端限制用于降低误操作风险，但该账号有能力改变会话设置，因此不能写成“数据库账号层不可绕过的强制只读”。本课只执行下面冻结的 `SELECT`，不使用 `UPDATE`、`DELETE`、`TRUNCATE` 或 DDL 做破坏性验证。

### 4.3 四组业务数据查询

> **本节要解决什么**：用四条已实测的 `SELECT`，把登录身份、P/T/C 范围、页面资料和三条 RAG 索引串成一条可观察的数据链。

&emsp;&emsp;下面四条 SQL 全部在已经通过 4.2 的 DBeaver 连接中执行，不是 Notebook 的 Python 单元。4.3.1 使用 `platform_identity_db`；其余三条使用 `platform_core_db`。切换连接或重新连接后先重新确认 4.2，只执行下方冻结的 `SELECT`，不查看 `password_hash`，不执行 `UPDATE`、`DELETE`、`TRUNCATE` 或 DDL（修改表结构的语句）。

> **【DBeaver 的边界】**：这里的 DBeaver 连接仅用于本机、脱敏的教学观察，帮助定位字段实际存放的位置；它不是产品用户读取业务数据的正常路径，也不能绕过 API、store 或 Gateway 的应用授权。即使查询能返回一行记录，也不能据此推断当前登录用户应当看到该记录。

&emsp;&emsp;本节使用 `admin`、成员 A（`perm_a_a6b5_r4`）与成员 B（`perm_b_a6b5_r4`）进行身份与数据记录对照。账号名称仅用于关联 UI 身份和数据库记录；课件不包含密码、连接串或环境变量内容。下列四条查询已于 2026-07-24 在当前 Docker 数据中实测返回结果；清理数据卷或重建 fixture 后，ID 和计数可能变化，应以自己的执行结果为准。

#### 4.3.1 身份账号与管理员标记

&emsp;&emsp;切换到 `platform_identity_db`。`users` 负责身份认证；本项目中后台管理员标记是 `is_admin`，它不是“可以读取所有私人正文”的万能权限。此查询只取身份 ID、账号、管理员标记和创建时间，刻意不取密码散列。

```sql
SELECT
  id,          -- 用户唯一身份 ID；用于与资源 owner_user_id 对照
  username,    -- 登录账号
  is_admin,    -- true=后台管理员；false=普通成员
  created_at   -- 账号创建时间
FROM public.users
WHERE username IN ('admin', 'perm_a_a6b5_r4', 'perm_b_a6b5_r4')
ORDER BY username;
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171629997.png" alt="DBeaver 中查询教学账号身份与管理员标记的结果" width=84%></div>

&emsp;&emsp;结果应显示三条教学账号记录，且只有 `admin` 的 `is_admin=true`。记录中的 `id` 是下一步 `owner_user_id` 的对照线索；它只能说明“资源归属给谁”，不能单独推出当前请求是否被授权。

#### 4.3.2 P/T/C 场景范围与资源归属

&emsp;&emsp;切换到已经完成 4.2 确认的 `platform_core_db` 连接。`scenarios` 是权限观察的主表：`id` 是场景唯一标识；`owner_user_id` 表示资源拥有者，`organization_id` 表示组织，`visibility` 保存 `private`、`team`、`company` 三种范围，`data->>'name'` 是 JSONB（PostgreSQL 的 JSON 文档字段）中的场景展示名称。

```sql
SELECT
  s.id AS scenario_id,         -- 场景唯一标识；用于关联和结果核对
  owner_user_id,              -- 创建者/私人资源的 owner
  organization_id,            -- 所属组织
  visibility,                 -- private、team、company 三种范围
  status,                     -- 当前资源状态，例如 ready
  data->>'name' AS scene_name -- JSONB 中的场景展示名称
FROM public.scenarios
WHERE data->>'name' IN (
  '成员A · 青屿仓库存校验',
  '团队 · 赤帆项目交接',
  '公司 · 雾桥计划客户图谱'
)
ORDER BY visibility, scene_name;
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171631911.png" alt="DBeaver 中查询 P/T/C 场景范围和资源归属的结果" width=84%></div>

&emsp;&emsp;当前 fixture 中三条场景属于同一组织，且都由成员 A 创建，因此三行 `owner_user_id` 都与成员 A 的身份 ID 对应。`scenario_id` 只用于资源关联和结果核对，不要复制账号、密码或资料正文。这不代表成员 A 独占团队或公司资料：`private` 还要检查 owner；`team` 与 `company` 先受组织和团队范围约束，再由 API / store 按当前动作作最终判断。

#### 4.3.3 页面资料与场景范围的关联

&emsp;&emsp;仍在 `platform_core_db`。`files` 通过 `scenario_id` 关联 `scenarios`；前台资料名称位于 `files.data.originalName`，不是独立的关系型列。查询把页面中看到的资料名称、场景名称、范围和 owner 放到同一行，便于从 UI 追到数据关系。

```sql
SELECT
  s.visibility,                              -- 资料的 P/T/C 范围
  f.data->>'originalName' AS document_name,  -- 前台资产页中的资料名称
  s.data->>'name' AS scene_name,             -- 资料归属的业务场景
  s.owner_user_id                            -- 创建者；私人资料需与当前成员匹配
FROM public.files AS f
JOIN public.scenarios AS s ON s.id = f.scenario_id
WHERE f.data->>'originalName' IN (
  '青屿仓库存校验单（成员A私人）.md',
  '赤帆项目交接清单（团队共享）.md',
  '雾桥计划客户关系图谱（公司知识）.md'
)
ORDER BY s.visibility;
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171629377.png" alt="DBeaver 中查询资料与 P/T/C 场景关联的结果" width=84%></div>

&emsp;&emsp;三份前台资料分别关联到 `private`、`team`、`company` 场景。这个关联解释“页面名称从哪里来、它属于哪个范围”；它不能替代第五章的授权判断，因为授权还要结合当前用户、组织、团队与动作。

#### 4.3.4 三条 RAG 链路的索引摘要

&emsp;&emsp;仍在 `platform_core_db`。`knowledge_objects` 记录资料已进入哪条 RAG 链路；这里按场景、RAG 引擎和索引对象类型聚合，只观察计数，不打开原始资料正文。`Gbrain` 是项目中 Nano Brain 链路的记录名称。

```sql
SELECT
  s.data->>'name' AS scene_name,  -- 业务场景
  k.rag_engine,                   -- Naive RAG、Gbrain、GraphRAG
  k.kind,                         -- 证据块、知识页或图对象
  COUNT(*) AS object_count        -- 当前索引对象数量
FROM public.knowledge_objects AS k
JOIN public.scenarios AS s ON s.id = k.scenario_id
WHERE s.data->>'name' IN (
  '成员A · 青屿仓库存校验',
  '团队 · 赤帆项目交接',
  '公司 · 雾桥计划客户图谱'
)
GROUP BY s.data->>'name', k.rag_engine, k.kind
ORDER BY scene_name, k.rag_engine;
```

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260724171632128.png" alt="DBeaver 中查询三条 RAG 索引对象汇总的结果" width=84%></div>

&emsp;&emsp;当前结果显示：私人资料有一条 `Naive RAG` 证据块；团队资料有一条 `Gbrain` 知识页；公司资料同时有 `GraphRAG` 图对象与 `Naive RAG` 证据块。`object_count=1` 只说明该教学资料已产生可观察索引对象，不能单独证明检索答案或 Agent 调用一定成功。

### 4.4 查询结果判读与证据边界

> **本节要解决什么**：区分“数据库中存在什么”和“业务用户被允许做什么”，并为异常结果选择正确的下一步。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>DBeaver 查询结果的解释边界</font></p>
<div class="center">

| 结果或查询 | 先检查什么 | 可以得出的结论 | 不能得出的结论 |
|--------|--------|--------|--------|
| `users` 返回记录 | 数据库与教学账号筛选条件 | 身份 ID、账号与管理员标记存在 | 管理员可以读取所有私人正文 |
| `scenarios` 返回记录 | owner、组织与 P/T/C 范围 | 当前资源的归属和声明范围可观察 | 任意成员都能读取该资源 |
| `files` + `scenarios` 返回记录 | 资料名与场景关联 | 页面资料属于哪个业务场景 | 前端已经完成授权判断 |
| `knowledge_objects` 返回记录 | RAG 引擎、对象类型与计数 | 对应链路存在可观察索引对象 | 检索或 Agent 一定返回该资料 |
| 查询返回空结果 | 数据库、fixture 与筛选条件 | 当前条件没有匹配的教学记录 | 数据库或系统一定损坏 |
| 连接被拒绝 | 2.6 观察覆盖与 DBeaver 连接信息 | 当前无法通过宿主机入口观察 | 默认 Compose 应永久公开数据库 |
| 只读状态不是 `on` | 4.2 初始化设置与当前连接 | 当前会话护栏未满足，应停止查询 | 可以先查询、稍后再修复 |

</div>

&emsp;&emsp;把本章理解为“事实定位”，不是“权限裁决”。DBeaver 中查到一行，只说明当前数据库账号能看到这条记录；它不会自动形成 Web 用户的 `UserContext`，也不会触发 API、store 或 Gateway 的授权流程。第五章才会把用户、owner、组织、团队和动作放进同一条规则，计算 `can_read` 与 `can_manage`。

&emsp;&emsp;如果教学 fixture 已被清理或重建，ID、行数和计数可能变化。此时先核对数据库和筛选条件，不要删除 `WHERE`、导出整张表或读取正文来“寻找数据”。截图只代表制作课件时的一次脱敏样本，自己的执行输出才是当前环境证据。

> **下一站**：保持 DBeaver 连接和第二章的观察端口，进入第五章查询用户权限与真实可读数据；全部 SQL 完成后再统一关闭观察端口。

## <center>第五章：用户权限与可读数据查询</center>

&emsp;&emsp;本章只围绕一条数据库查询链展开：用户名确定 `user_id`，`user_id` 形成当前用户上下文，再与场景的 owner、组织、团队和 `scope`（资源可见范围）比较，最终得到该用户能读取和管理哪些资源，以及这些资源关联的任务、文件和知识对象。

&emsp;&emsp;当前项目没有启用 PostgreSQL RLS（Row-Level Security，数据库行级安全策略），数据库也没有单独的“用户—资源权限表”。真实授权由 `packages/platform/src/platform-store.ts` 的 `canAccess` 和 `canManageScenario` 在应用层计算。因此，本章 SQL 的作用是**按照源码规则复现权限判断并观察结果**，不是把数据库服务账号包装成业务用户。

&emsp;&emsp;身份库当前只保存 `id`、`username` 和 `is_admin`。组织与团队尚未写入 `users` 表；应用会通过 `packages/platform/src/org-team-defaults.ts` 补充默认上下文：普通成员属于 `org_companybrain / default-team`，管理员属于 `org_companybrain / platform-admin`。

> **【常见误解与真实机制】**：能在 DBeaver 中查到一行数据，不等于当前业务用户已经获得读取权限。DBeaver 使用的是数据库服务账号；本章先用 SQL 按源码规则计算业务权限，再用 `can_read` 控制是否返回关联数据。线上请求仍由应用层执行真实授权。

> **【本章自学地图】**：查询用户身份 → 填入用户上下文 → 计算 `can_read` 与 `can_manage` → 查看可读任务、文件和知识对象 → 更换用户重复比较。

### 5.1 用户身份查询

> **本节要解决什么**：从身份库取得真实 `user_id`、管理员标记和当前应用默认的组织与团队上下文。

&emsp;&emsp;继续使用第四章的 DBeaver 观察入口。连接 `platform_identity_db` 后，先按 4.2 重新确认当前会话只读，再执行下面的查询。`app_default_organization_id` 和 `app_default_team_id` 是根据当前源码补出的应用默认值，不是 `users` 表的物理字段。

```sql
SELECT
    id AS user_id,  -- 用户唯一 ID；用于和资源 owner_user_id 对照
    username,       -- 登录账号；用于确认当前查询的是哪个教学用户
    is_admin,       -- 管理员标记；true=管理员，false=普通成员
    'org_companybrain' AS app_default_organization_id, -- 应用补充的默认组织，不是 users 物理字段
    CASE
        WHEN is_admin THEN 'platform-admin'
        ELSE 'default-team'
    END AS app_default_team_id -- 应用补充的默认团队，不是 users 物理字段
FROM public.users              -- 身份库中的用户账号表
WHERE username IN (            -- 只查询本节使用的三个教学账号
    'perm_a_a6b5_r4',
    'perm_b_a6b5_r4',
    'admin'
)
ORDER BY username;             -- 固定显示顺序，便于对照查询结果
```

&emsp;&emsp;当前教学数据的查询结果如下。若数据库被重新初始化，UUID（通用唯一标识符）可能变化，应始终以自己刚执行的查询结果为准。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>教学账号的用户上下文</font></p>
<div class="center">

| username | user_id | is_admin | 应用默认组织 | 应用默认团队 |
|--------|--------|--------|--------|--------|
| `admin` | `eca7f26e-9055-4ebc-97fe-59ab94c94806` | true | `org_companybrain` | `platform-admin` |
| `perm_a_a6b5_r4` | `1269a895-b09e-4fe7-b022-a76365d5668c` | false | `org_companybrain` | `default-team` |
| `perm_b_a6b5_r4` | `a6186957-587d-4e76-ab35-03c4e298a85e` | false | `org_companybrain` | `default-team` |

</div>

&emsp;&emsp;本章先选择成员 B。复制 `perm_b_a6b5_r4` 的 `user_id`、`is_admin`、组织和团队，填入下一节 SQL 顶部的 `current_user_context`。这一步建立的是业务用户上下文，不要把它与 4.2 中 DBeaver 当前使用的 PostgreSQL 数据库账号混淆。

### 5.2 用户权限与可读数据

> **本节要解决什么**：使用一个真实 `user_id`，同时查看场景读取权限、管理权限以及允许读取的任务、文件和知识对象。

&emsp;&emsp;在 DBeaver 中切换到 `platform_core_db`，先按 4.2 重新确认这个连接的会话只读，再执行下面的 SQL。最上方四个参数来自 5.1；本例已经填入成员 B。`current_user_context` 只是 SQL 中手工构造的业务用户上下文，不会改变 DBeaver 的数据库身份。`scenario_access` 只提取权限判断需要的资源字段；`permission_result` 按源码顺序计算 `can_read` 和 `can_manage`；最后三个子查询只在 `can_read=true` 时返回关联数据，避免把拒绝访问的文件名或知识对象暴露出来。

```sql
WITH current_user_context AS ( -- 当前请求者上下文；四个值来自 5.1
    SELECT
        'a6186957-587d-4e76-ab35-03c4e298a85e'::TEXT AS user_id, -- 成员 B 的用户 ID
        FALSE AS is_admin,                                     -- 成员 B 不是管理员
        'org_companybrain'::TEXT AS organization_id,           -- 当前用户所属组织
        ARRAY['default-team']::TEXT[] AS team_ids              -- 当前用户所属团队列表
),

scenario_access AS ( -- 提取场景权限判断需要的资源字段
    SELECT
        s.id AS scenario_id,             -- 场景唯一 ID
        s.data->>'name' AS scene_name,   -- 前台显示的场景名称
        s.status,                        -- 场景当前状态，例如 ready
        COALESCE(
            s.data->'accessControl'->>'scope',
            s.visibility
        ) AS scope,                      -- 可见范围：private、team 或 company
        COALESCE(
            s.data->'accessControl'->>'ownerUserId',
            s.owner_user_id
        ) AS owner_user_id,              -- 资源拥有者的用户 ID
        COALESCE(
            s.data->'accessControl'->>'organizationId',
            s.organization_id
        ) AS organization_id,            -- 资源所属组织
        COALESCE(
            s.data->'accessControl'->'teamIds',
            '[]'::JSONB
        ) AS team_ids                    -- 允许共享的资源团队列表
    FROM public.scenarios AS s           -- 平台核心库中的业务场景表
    WHERE s.data->>'name' IN (           -- 只观察本节 P/T/C 三个教学场景
        '成员A · 青屿仓库存校验',
        '团队 · 赤帆项目交接',
        '公司 · 雾桥计划客户图谱'
    )
),

permission_result AS ( -- 将当前用户与每个场景进行权限比较
    SELECT
        a.*,                              -- 保留前一个 CTE 的场景权限字段
        u.user_id AS current_user_id,     -- 本次权限判断使用的用户 ID

        CASE
            WHEN u.is_admin THEN TRUE     -- 管理员可以读取知识资源
            WHEN a.scope = 'private'
                THEN a.owner_user_id = u.user_id -- private 只允许 owner
            WHEN a.owner_user_id = u.user_id THEN TRUE -- owner 可读取自己的资源
            WHEN a.organization_id <> u.organization_id THEN FALSE -- 跨组织拒绝
            WHEN a.scope = 'company' THEN TRUE -- 同组织 company 资源可读
            WHEN a.scope = 'team' THEN EXISTS (
                SELECT 1
                FROM jsonb_array_elements_text(a.team_ids)
                    AS resource_team(team_id)
                WHERE resource_team.team_id = ANY(u.team_ids) -- 用户与资源团队存在交集
            )
            ELSE FALSE                    -- 未命中规则时默认拒绝
        END AS can_read,                  -- 当前用户能否读取该场景

        (
            u.is_admin
            OR a.owner_user_id = u.user_id
        ) AS can_manage,                  -- 只有管理员或 owner 可以管理场景

        CASE
            WHEN u.is_admin
                THEN '允许：管理员资源读取规则'
            WHEN a.scope = 'private'
                 AND a.owner_user_id = u.user_id
                THEN '允许：当前用户是 private 资源 owner'
            WHEN a.scope = 'private'
                THEN '拒绝：private 资源仅 owner 可读'
            WHEN a.owner_user_id = u.user_id
                THEN '允许：当前用户是资源 owner'
            WHEN a.organization_id <> u.organization_id
                THEN '拒绝：用户与资源不属于同一组织'
            WHEN a.scope = 'company'
                THEN '允许：同组织 company 资源'
            WHEN a.scope = 'team'
                 AND EXISTS (
                     SELECT 1
                     FROM jsonb_array_elements_text(a.team_ids)
                         AS resource_team(team_id)
                     WHERE resource_team.team_id = ANY(u.team_ids)
                 )
                THEN '允许：同组织且团队存在交集'
            WHEN a.scope = 'team'
                THEN '拒绝：用户与资源没有团队交集'
            ELSE '拒绝：未命中可读规则'
        END AS read_reason                -- can_read 对应的中文判定原因

    FROM scenario_access AS a
    CROSS JOIN current_user_context AS u  -- 用同一用户检查三个教学场景
)

SELECT
    p.current_user_id,  -- 当前请求者的用户 ID
    p.scenario_id,      -- 被检查的场景 ID
    p.scene_name,       -- 被检查的场景名称
    p.scope,            -- 场景可见范围
    p.owner_user_id,    -- 场景 owner 的用户 ID
    p.organization_id,  -- 场景所属组织
    p.team_ids,         -- 场景允许共享的团队列表
    p.status,           -- 场景当前状态
    p.can_read,         -- 当前用户是否可以读取
    p.can_manage,       -- 当前用户是否可以管理
    p.read_reason,      -- 允许或拒绝的具体原因

    CASE WHEN p.can_read THEN (
        SELECT COALESCE(
            jsonb_agg(
                jsonb_build_object(
                    'title', t.data->>'title', -- 任务标题
                    'status', t.status         -- 任务状态
                )
                ORDER BY t.created_at
            ),
            '[]'::JSONB
        )
        FROM public.tasks AS t
        WHERE t.scenario_id = p.scenario_id
    ) END AS readable_tasks, -- 允许读取时返回场景任务；拒绝时返回 NULL

    CASE WHEN p.can_read THEN (
        SELECT COALESCE(
            jsonb_agg(
                f.data->>'originalName' -- 前台显示的文件原始名称
                ORDER BY f.uploaded_at
            ),
            '[]'::JSONB
        )
        FROM public.files AS f
        WHERE f.scenario_id = p.scenario_id
          AND f.deleted_at IS NULL
    ) END AS readable_files, -- 允许读取时返回未删除文件；拒绝时返回 NULL

    CASE WHEN p.can_read THEN (
        SELECT COALESCE(
            jsonb_agg(
                jsonb_build_object(
                    'title', k.data->>'title', -- 知识对象标题
                    'rag_engine', k.rag_engine, -- 对应的 RAG 引擎
                    'kind', k.kind              -- 索引对象类型
                )
                ORDER BY k.created_at
            ),
            '[]'::JSONB
        )
        FROM public.knowledge_objects AS k
        WHERE k.scenario_id = p.scenario_id
    ) END AS readable_knowledge_objects -- 允许读取时返回知识对象；拒绝时返回 NULL

FROM permission_result AS p
ORDER BY
    CASE p.scope                    -- 按 private、team、company 排序
        WHEN 'private' THEN 1
        WHEN 'team' THEN 2
        WHEN 'company' THEN 3
        ELSE 4
    END,
    p.scene_name;                   -- 同范围内按场景名称排序
```

&emsp;&emsp;这条 SQL 同时回答三个问题：`can_read` 表示当前用户能否读取场景；`can_manage` 表示当前用户能否管理场景；三个 `readable_*` 字段展示允许读取时实际关联的任务、文件和知识对象。这样学员不需要再执行第三条 SQL，也能从同一张结果表看到“权限结论”和“可读数据”。

### 5.3 查询结果判读

> **本节要解决什么**：从同一张查询结果中解释成员 B 为什么能读 team/company，却不能读 private，也不能管理 Alice 的场景。

&emsp;&emsp;当前教学数据中，成员 A 是三个 P/T/C（private、team、company）场景的 owner，成员 B 与成员 A 同组织、同团队但不是 owner。第二条 SQL 的真实运行结果可以压缩为下面三行。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>成员 B 的场景权限结果</font></p>
<div class="center">

| 场景 | scope | can_read | can_manage | read_reason |
|--------|--------|--------|--------|--------|
| 成员A · 青屿仓库存校验 | private | false | false | private 资源仅 owner 可读 |
| 团队 · 赤帆项目交接 | team | true | false | 同组织且团队存在交集 |
| 公司 · 雾桥计划客户图谱 | company | true | false | 同组织 company 资源 |

</div>

&emsp;&emsp;这三行展示了用户权限中最容易混淆的两条规则。读取权限由 owner、组织、团队和 `scope` 共同决定；管理权限只允许有效管理员或资源 owner。因此，成员 B 能读取成员 A 的 team/company 资料，但不能管理这些场景。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>成员 B 实际可读的关联数据</font></p>
<div class="center">

| 场景 | readable_tasks | readable_files | readable_knowledge_objects |
|--------|--------|--------|--------|
| 成员A · 青屿仓库存校验 | `NULL` | `NULL` | `NULL` |
| 团队 · 赤帆项目交接 | 1 条任务，`status=ready` | `赤帆项目交接清单（团队共享）.md` | `Gbrain / knowledge_page` |
| 公司 · 雾桥计划客户图谱 | 1 条任务，`status=ready` | `雾桥计划客户关系图谱（公司知识）.md` | `GraphRAG / graph_object`；`Naive RAG / evidence_chunk` |

</div>

&emsp;&emsp;第二张表不是另外执行的查询，而是同一次 5.2 结果中三个 `readable_*` 列的整理。任务标题含有教学 fixture（固定样本）的批次后缀，重新准备样本后可能变化，因此判读时重点观察任务数量与状态、文件名、RAG 引擎和对象类型。数据库里存在 private 记录，也不表示成员 B 应看到它关联的文件名和知识对象，所以 `can_read=false` 时三个字段都返回 `NULL`。

&emsp;&emsp;要观察成员 A 或管理员，不需要新增 SQL，只替换 `current_user_context` 的四个参数后重新运行 5.2：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>更换用户后的预期权限</font></p>
<div class="center">

| 用户 | private | team | company | 管理权限 |
|--------|--------|--------|--------|--------|
| 成员 A | 可读 | 可读 | 可读 | 三个场景都是 owner，可管理 |
| 成员 B | 不可读 | 可读 | 可读 | 不是 owner，不可管理 |
| Admin | 可读 | 可读 | 可读 | 有效管理员，可管理 |

</div>

### 5.4 权限查询边界

> **本节要解决什么**：明确两条 SQL 已经覆盖哪些数据，以及哪些内容不应通过只读权限查询展示。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>权限 SQL 的数据覆盖范围</font></p>
<div class="center">

| 层 | 已看到的数据 | 为什么需要 |
|--------|--------|--------|
| 用户身份 | `user_id`、username、`is_admin` | 确认当前请求者是谁 |
| 用户上下文 | 默认组织与团队 | 为 team/company 范围提供比较条件 |
| 场景权限 | owner、scope、组织、团队、`can_read`、`can_manage` | 解释允许或拒绝 |
| 可读业务数据 | 任务标题与状态、文件名、知识对象标题、RAG 引擎和对象类型 | 证明“可读”最终对应哪些真实数据 |

</div>

&emsp;&emsp;站在学员角度，这两条 SQL 已经形成完整闭环，不需要再增加第三条查询。第一条回答“当前用户是谁”；第二条回答“这个用户能读什么、能否管理，以及允许读取时能看到哪些任务、文件和知识对象”。继续增加会话、审计日志、全文内容或跨库导出，会把课程从用户资源权限带到新的主题。

&emsp;&emsp;下面三类内容应明确保持不可见：`password_hash` 和 session Token 属于身份凭据；文件正文、聊天正文和模型工具载荷不属于权限摘要；`readable_*` 在 `can_read=false` 时必须为空，不能为了展示效果绕过条件。

> **【源码锚点】**：`packages/platform/src/platform-store.ts` 的 `canAccess` 决定场景读取范围，`canManageScenario` 只允许有效管理员或 owner 管理场景；`packages/platform/src/org-team-defaults.ts` 提供当前默认组织与团队。SQL 按同样顺序计算，但线上请求仍以应用接口的真实判断为准。

### 5.5 学习检查

> **本节要解决什么**：确认自己能从查询结果解释用户、资源和权限之间的关系。

1. **为什么成员 B 看不到成员 A 的 private 场景？** 因为 private 资源只允许 owner 读取。

2. **为什么成员 B 可以读取 team 场景？** 因为双方组织相同，并且 `team_ids` 存在交集。

3. **为什么成员 B 可以读取却不能管理 team 场景？** 读取规则允许团队共享，管理规则仍要求管理员或 owner。

4. **为什么 `readable_*` 不能在 `can_read=false` 时返回数据？** 因为权限摘要本身不能泄露被拒绝资源的文件名、任务和知识对象。

&emsp;&emsp;至此，本节形成了一条最小、直观、可重复的用户权限查询链：`username → user_id → 用户上下文 → 资源访问字段 → can_read / can_manage → 可读业务数据`。以后更换用户时，只需要重新执行 5.1，并替换 5.2 顶部四个参数。

### 5.6 临时观察端口关闭

> **本节要解决什么**：完成 Neo4j、DBeaver 与权限查询后，收回数据库的本机观察端口，同时保持业务服务和数据卷不变。

&emsp;&emsp;关闭 DBeaver 或 Neo4j Browser 不会改变 Docker 端口。要收回观察覆盖，必须重新使用不含 `compose.dev-ports.yml` 的显式两文件组合收敛同一个 Compose 项目。这里复用 2.3 已经构建的镜像，不重新构建，也不删除容器数据卷。

In [ ]:
# 回到默认文件集；不加载 compose.dev-ports.yml
!docker compose --project-name ff-companybrain --env-file "$FF_REPO/deploy/compose/.env" -f "$FF_REPO/deploy/compose/compose.student.yml" -f "$FF_REPO/deploy/compose/compose.build.yml" up --detach

&emsp;&emsp;默认配置重新生效后，再用同一文件组合读取 Compose 状态。检查重点不是“命令是否退出为 0”，而是 Web 仍保留产品入口，PostgreSQL 与 Neo4j 不再显示宿主机端口映射。

In [ ]:
# 只查看服务和端口；不修改容器或数据
!docker compose --project-name ff-companybrain --env-file "$FF_REPO/deploy/compose/.env" -f "$FF_REPO/deploy/compose/compose.student.yml" -f "$FF_REPO/deploy/compose/compose.build.yml" ps

&emsp;&emsp;正确结果是 Web 入口继续存在，PostgreSQL 的 `15432`、Neo4j 的 `17474` 与 `17687` 不再出现在宿主机映射中。以后如需重新观察，先再次执行 2.6；完成后仍回到本节关闭观察端口。

## <center>第六章：四课合流——从知识入库到可信回答</center>

&emsp;&emsp;四节课完成的不是四套彼此独立的功能，而是一条受控知识链：第一课让文档进入 Native/Traditional RAG，第二课以 GraphRAG 与 Nano Brain/nano-Gbrain 补齐关系知识和可治理知识，第三课用 Agent 编排一次全域问答，第四课把 Compose、存储与产品权限落到可检查的边界。任何一环都不能替另一环背书：数据库中有记录，不等于用户有权读取；候选相关，不等于结论正确；容器启动，也不等于检索或 Agent 端到端已经通过。

### 6.1 四节课完成的是同一套系统

> **本节要解决什么**：用最小表格收束四节课分别解决的系统问题，而不重复前三课的代码、性能数字或运行步骤。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四课如何汇成同一套系统</font></p>
<div class="center">

| 课程 | 解决的系统问题 | 留下的关键产物 | 不能替代 |
|--------|----------------|----------------|----------|
| 第一课 | 文档怎样成为可检索知识 | Native/Traditional RAG 的文档、表格与检索基础 | 图关系与用户权限判断 |
| 第二课 | 不同知识结构怎样加工 | GraphRAG 的关系证据；Nano Brain 的可治理知识产物 | 跨链路候选的统一比较 |
| 第三课 | 一次全域问答怎样被组织 | Agent 会话、复合检索、记忆与运行记录 | 模块内部的 RAG 实现 |
| 第四课 | 服务和数据怎样在边界内运行 | Compose、PostgreSQL/Neo4j、用户与资源权限证据 | 把观察到的数据直接交给任意用户 |

</div>

&emsp;&emsp;**常见误解**：把四节课顺序学完，后面的回答就天然可信。**真实机制**：可信来自知识加工、候选比较、编排、运行边界和业务权限共同成立；缺少其中任何一项，都应缩小结论范围。

### 6.2 从知识进入到可信回答

> **本节要解决什么**：沿一条端到端逻辑链，理解哪些地方独立加工，哪些地方合流，以及为何回答仍需受权限和证据约束。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260727165000089.png" alt="ff-companybrain 从知识入库到可信回答的最终系统架构图" width=92%></div>

> **【读图边界】**：图展示当前职责、默认网络边界与主要存储关系；它不是对摄取、检索、外部模型调用或 Agent 端到端成功的声明。默认只有 Web 对宿主机开放；临时数据库观察入口不属于默认产品路径。

&emsp;&emsp;资料与场景进入平台后，分别交给适合的 RAG 链路独立加工：Native/Traditional RAG 面向文档与表格，Nano Brain 面向可治理知识，GraphRAG 同时使用 PostgreSQL 辅助数据和 Neo4j 图关系。用户请求从 Web 进入：API 负责鉴权和平台分发；全域 Agent 会话通过 `company_knowledge_search` 让平台完成意图路由、多路候选聚合与统一 rerank（重排序：用同一标准比较不同候选），模型再依据结构化证据组织回答。用户、owner、组织、团队和 scope 的权限边界先约束哪些候选可用于当前请求；随后才保存本轮授权范围内的 citations、run 与 checkpoint。图聚焦全域 profile；普通 non-global Agent profile 使用 native Tool 调用受保护模块 HTTP，两条路径都不直连模块数据库。

&emsp;&emsp;**常见误解**：三条 RAG 的数据或原始分数已经合并成一个总库，由 Agent 自己任选结果。**真实机制**：三条 RAG 独立存储、独立加工、独立召回；合流发生在候选与证据层，统一 rerank 负责相关性比较。GraphRAG 的 PostgreSQL 与 Neo4j 也不能互相替代；它们分别保存不同职责的数据。

&emsp;&emsp;至此，`ff-companybrain` 项目完成收尾。四节课已经将三条 RAG、Agent 编排、Docker Compose 部署、PostgreSQL/Neo4j 存储与权限边界串成一套完整的企业级知识中台。